In [21]:
# 设置环境变量  
!export API_KEY="ms-65a65454-0502-4762-961b-e238adcc56c6"  
!export API_BASE="https://api-inference.modelscope.cn/v1"  # 或您的模型API端点  
  
# 安装依赖（如果尚未安装）  
%pip install openai aiohttp transformers torch datasets peft accelerate wandb


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [18]:
# 选择数据集（三选一）  
DATASET="scierc"  # 可选: genwiki, rebel_sub, scierc  
  
# 创建数据目录（如果不存在）  
!mkdir -p ./datasets/GPT4o_mini_result_${DATASET}  
!mkdir -p ./graph_judger/data/${DATASET}

In [20]:
!echo "=== 开始ECTD阶段 ==="  
  
# 步骤1: 实体提取和文本去噪  
!echo "运行实体提取..."  
!python ./chat/run_chatgpt_entity.py  
  
# 步骤2: 三元组生成  
!echo "运行三元组生成..."  
!python ./chat/run_chatgpt_triple.py  
  
!echo "ECTD阶段完成，结果保存在 ./datasets/GPT4o_mini_result_${DATASET}/"

=== 开始ECTD阶段 ===
运行实体提取...


Traceback (most recent call last):
  File "/workspaces/gg/./chat/run_chatgpt_entity.py", line 2, in <module>
    from chat.model_interface import api_model  
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'chat'
运行三元组生成...
Traceback (most recent call last):
  File "/workspaces/gg/./chat/run_chatgpt_triple.py", line 2, in <module>
    from chat.model_interface import api_model  
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'chat'
ECTD阶段完成，结果保存在 ./datasets/GPT4o_mini_result_/


In [ ]:
!echo "=== 开始KASFT阶段 ==="  
  
# 步骤1: 生成训练指令（需要手动运行Jupyter notebook）  
!echo "请在Jupyter中运行: ./datasets/prepare_KGCom.ipynb 的第一个单元格"  
!echo "完成后按任意键继续..."  
!read -n 1 -s  
  
# 步骤2: LoRA微调  
!echo "开始LoRA微调..."  
!cd ./graph_judger  
!python lora_finetune_${DATASET}_context.py  
!cd ..  
  
!echo "KASFT阶段完成，模型保存在 ./graph_judger/models/"

In [ ]:
!echo "=== 开始GJ阶段 ==="  
  
# 步骤1: 生成测试数据（需要手动运行Jupyter notebook）  
!echo "请在Jupyter中运行: ./datasets/prepare_KGCom.ipynb 的第二个单元格"  
!echo "完成后按任意键继续..."  
!read -n 1 -s  
  
# 步骤2: 批量推理  
!echo "开始批量推理..."  
!cd ./graph_judger  
!python lora_infer_batch.py  
!cd ..  
  
# 步骤3: 三元组过滤（需要手动运行Jupyter notebook）  
!echo "请在Jupyter中运行: ./datasets/prepare_KGCom.ipynb 的第三个单元格"  
!echo "完成后按任意键继续..."  
!read -n 1 -s  
  
!echo "GJ阶段完成，最终结果保存在 ./datasets/GPT4o_mini_result_${DATASET}/Graph_Iteration1/test_generated_graphs_final.txt"

In [ ]:
!echo "=== 开始评估阶段 ==="  
  
# 修改评估脚本路径并运行  
!cd ./graph_evaluation  
# 请确保eval.sh中的路径指向正确的数据集和结果文件  
!bash eval.sh  
!cd ..  
  
!echo "Demo运行完成！"